# Preprocesamiento

Primero se repasó el paper asociado al dataset, puede ser encontrado en este [enlace](https://openaccess.thecvf.com/content_CVPR_2019/papers/Bergmann_MVTec_AD_--_A_Comprehensive_Real-World_Dataset_for_Unsupervised_Anomaly_CVPR_2019_paper.pdf). En el se comenta como en el area de detección de anomalías no se tenía un dataset de uso común como en otras área de machine learning como en detección de objetos con ImageNet y MNIST. Por lo que se creó este conjunto de imágenes como un precedente del nicho.

El dataset cuenta con 15 categorías, 3629 imágenes de training y validación, y 1725 para testing. De dichas categorías nostros solo estamos interasados en las siguientes:

| Categoría | Training | Testing | Resolución|
|-----------|----------|---------|-------|
| Cable     |       224|      150|   1024|
| Capsule   |       219|      132|   1000|
| Screw     |       320|      160|   1024|
| Transistor|       213|      100|   1024|


Por otro lado, el dataset ya trae definidas las particiones de entrenamiento y testing. Por lo que solo debemos extraer un porcentaje del de training de cada una y con eso formar el de validation. En nuestro caso tomaremos un 15% de cada clase y así quedaran las particiones finales: 

| Categoría | Validation (15%) | Training  | Testing | Resolución |
|-----------|------------------|----------------|---------|---------------------|
| Cable      |  33 | 191 | 150 | 1024 |
| Capsule    |  32 | 187 | 132 | 1000 |
| Screw      |  48 | 272 | 160 | 1024 |
| Transistor |  31 | 182 | 100 | 1024 |


Por lo que tendrémos que extraerlas y preprocesarlos para dejarlas en tensores de 128x128x3.


In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [29]:
from pathlib import Path
import os

dataset_path ="/content/drive/MyDrive/Tarea3/dataset"
data_dir = Path(dataset_path)

train_cable_dir = data_dir / "cable" / "train" / "good"
train_capsule_dir = data_dir / "capsule" / "train" / "good"
train_screw_dir = data_dir / "screw" / "train" /"good"
train_transistor_dir = data_dir / "transistor" / "train" / "good"

In [30]:
train_transistor_dir.exists()

True

El proceso de conversion a tensores en este caso es fácil, tan solo se debe usar ToTensor de torchvision y eso ya nos dejará las imágenes como tensores normalizados. También para reescalarlas a 128x128 usamos el transform de Resize.

## Dataset de training y validation 

Primero trabajemos con el dataset de training ya que este solo cuenta con ejemplos "good", el proceso de lectura de los datasets de trainining será un tanto distinto

In [61]:
import torch
from PIL import Image
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((128, 128)), # Esto pasa las imágenes de la resu original a la que pide el profe
    transforms.ToTensor() # ToTensor para que si normalice
])


def load_training_dataset(dir):
    result = []
    
    for file in os.listdir(dir):
        path = os.path.join(dir, file)
        img = Image.open(path).convert("RGB")

        tensor = transform(img)
        result.append(tensor)

    return result

In [39]:

train_cable_list = load_training_dataset(train_cable_dir)
train_capsule_list = load_training_dataset(train_capsule_dir)
train_screw_list = load_training_dataset(train_screw_dir)
train_transistor_list = load_training_dataset(train_transistor_dir)

In [40]:
train_cable_tensor = torch.stack(train_cable_list)
train_capsule_tensor = torch.stack(train_capsule_list)
train_screw_tensor = torch.stack(train_screw_list)
train_transistor_tensor = torch.stack(train_transistor_list)

In [43]:
print(train_cable_tensor.shape)
print(train_capsule_tensor.shape)
print(train_screw_tensor.shape)
print(train_transistor_tensor.shape)

torch.Size([224, 3, 128, 128])
torch.Size([219, 3, 128, 128])
torch.Size([320, 3, 128, 128])
torch.Size([213, 3, 128, 128])


In [48]:
def split_train_val(X, val_ratio=0.15, seed=42):
    n = len(X)

    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(n, generator=generator)

    n_val = int(n * val_ratio)

    val_indices = indices[:n_val]
    train_indices = indices[n_val:]

    X_train = X[train_indices]
    X_val = X[val_indices]

    return X_train, X_val

In [49]:
# Ya acaá ahora si queda dividido en training y validation
cable_train, cable_val = split_train_val(train_cable_tensor, val_ratio=0.15)
capsule_train, capsule_val = split_train_val(train_capsule_tensor, val_ratio=0.15)
screw_train, screw_val = split_train_val(train_screw_tensor, val_ratio=0.15)
transistor_train, transistor_val = split_train_val(train_transistor_tensor, val_ratio=0.15)

In [54]:
X_train = torch.cat([
    cable_train,
    capsule_train,
    screw_train,
    transistor_train
], dim=0)

X_val = torch.cat([
    cable_val,
    capsule_val,
    screw_val,
    transistor_val
], dim=0)

In [58]:
print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: torch.Size([832, 3, 128, 128])
Val: torch.Size([144, 3, 128, 128])


In [59]:
# También creemos tensores "Label", no se usarán para entrenar, son solo metadata para que luego los gráficos muestren quien es cada clase

CLASS_TO_IDX = {
    "cable": 0,
    "capsule": 1,
    "screw": 2,
    "transistor": 3
}

y_train = torch.cat([
    torch.full((len(cable_train),), CLASS_TO_IDX["cable"]),
    torch.full((len(capsule_train),), CLASS_TO_IDX["capsule"]),
    torch.full((len(screw_train),), CLASS_TO_IDX["screw"]),
    torch.full((len(transistor_train),), CLASS_TO_IDX["transistor"]),
])

y_val = torch.cat([
    torch.full((len(cable_val),), CLASS_TO_IDX["cable"]),
    torch.full((len(capsule_val),), CLASS_TO_IDX["capsule"]),
    torch.full((len(screw_val),), CLASS_TO_IDX["screw"]),
    torch.full((len(transistor_val),), CLASS_TO_IDX["transistor"]),
])

In [60]:
print("Train:", y_train.shape)
print("Val:", y_val.shape)

Train: torch.Size([832])
Val: torch.Size([144])


## Dataset de testing

Este se debe manejar un tanto distinto debido a que en este si se tienen clases de defectos. Por lo que se seguirá la misma idea anterior de generar un tensor con todos los tensores correspondientes a la imágen de Testing, y también la creación de un tensor Y con metadata. Solo que en este caso el tensor de metadata no solo contendrá la clase de la imágen, si no que también el tipo de defecto. Cabe recalcar que los tipos de defecto varían en cada clase, por lo que el encoding usado para representarlos dependerá de la clase.  

In [ ]:
data_dir = Path(dataset_path)

test_cable_dir = data_dir / "cable" / "test"
test_capsule_dir = data_dir / "capsule" / "test" 
test_screw_dir = data_dir / "screw" / "test" 
test_transistor_dir = data_dir / "transistor" / "test" 

In [ ]:
def load_testing_dataset(dir, object_class):
    
    X_result = []
    y_result = []
    defect_mapping = {}

    # El index 0 está reservado para la clase good
    i = 1
    for defect_dir in dir.iterdir():

        if not defect_dir.is_dir():
            continue

        defect_type = defect_dir.name
        
        defect_code = 0 if defect_type == "good" else i

        defect_mapping[defect_type] = defect_code
        
        for file in os.listdir(defect_dir):
            path = os.path.join(defect_dir, file)
            img = Image.open(path).convert("RGB")

            img_tensor = transform(img)

            X_result.append(img_tensor)
            y_result.append(torch.tensor([object_class, defect_code]))
        
        # No sumamos en la iteracion de good
        i = i if defect_code == 0 else i+1
        
    return X_result, y_result, defect_mapping



In [ ]:
test_cable_list,      y_cable_list,      cable_mapping      = load_testing_dataset(test_cable_dir, 0)
test_capsule_list,    y_capsule_list,    capsule_mapping    = load_testing_dataset(test_capsule_dir, 1)
test_screw_list,      y_screw_list,      screw_mapping      = load_testing_dataset(test_screw_dir, 2)
test_transistor_list, y_transistor_list, transistor_mapping = load_testing_dataset(test_transistor_dir, 3)